# Can we predict a warranty claim within a year?
Target: `claim_within_year` (~12.6% positives).

Design notes (from checklist.md and columns.csv):
- `device_serial` is an identifier: dropped. Every serial occurs once, so rows are distinct entities and a row-level split is also an entity-level split.
- There is no date column, so no "later rows" ordering can be established; I use a stratified random split and assume future devices resemble these. This is a limitation.
- All other columns (channel, age, price, daily hours) are recorded at sale/registration, so are known at prediction time.

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, confusion_matrix
df = pd.read_csv('data.csv')
print(df.shape, df.device_serial.is_unique, df.isna().sum().sum())
df = df.drop(columns='device_serial')
df.describe(include='all')

## 1. Test set set aside before any modelling (20%, untouched until the end)

In [ ]:
y = df.claim_within_year; X = df.drop(columns='claim_within_year')
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)
print(len(X_tr), len(X_te), y_tr.mean(), y_te.mean())

## 2. Candidates named up front (before scoring any)
1. `baseline`: predicts the training prior (trivial).
2. `logreg`: standardised numerics + one-hot channel, logistic regression (C=1).
3. `logreg_C0.1`: same, stronger regularisation.
4. `hgb`: histogram gradient boosting (shallow, lr 0.05).

Selection metric: CV average precision (PR-AUC), since positives are rare. The threshold is then chosen on out-of-fold predictions by maximising F1. Preprocessing lives inside the pipelines, so it is fit within each fold.

In [ ]:
num = ['device_age_years', 'price_eur', 'daily_hours']; cat = ['channel']
def pre(scale=True):
    return ColumnTransformer([('n', StandardScaler() if scale else 'passthrough', num),
                              ('c', OneHotEncoder(handle_unknown='ignore'), cat)])
models = {
 'baseline': Pipeline([('p', pre()), ('m', DummyClassifier(strategy='prior'))]),
 'logreg': Pipeline([('p', pre()), ('m', LogisticRegression(C=1, max_iter=1000))]),
 'logreg_C0.1': Pipeline([('p', pre()), ('m', LogisticRegression(C=0.1, max_iter=1000))]),
 'hgb': Pipeline([('p', pre(False)), ('m', HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=0))]),
}
cv = StratifiedKFold(5, shuffle=True, random_state=1)
oof = {}; rows = []
for k, m in models.items():
    oof[k] = cross_val_predict(m, X_tr, y_tr, cv=cv, method='predict_proba')[:, 1]
    rows.append((k, roc_auc_score(y_tr, oof[k]), average_precision_score(y_tr, oof[k])))
cvres = pd.DataFrame(rows, columns=['model', 'cv_auc', 'cv_ap']).set_index('model'); cvres

## 3. Choose model and threshold on CV only

In [ ]:
best = cvres.cv_ap.drop('baseline').idxmax()
ths = np.linspace(0.02, 0.6, 59)
f1s = [f1_score(y_tr, oof[best] >= t) for t in ths]
thr = float(ths[int(np.argmax(f1s))])
print(best, 'threshold', thr, 'cv F1', max(f1s))

**Recorded choice (from CV only, before the test set is touched):** the model printed as `best` above with the printed F1-maximising threshold; features channel, device_age_years, price_eur, daily_hours.

## 4. Refit on all training rows; score the test set once

In [ ]:
final = models[best].fit(X_tr, y_tr)
p = final.predict_proba(X_te)[:, 1]
base = models['baseline'].fit(X_tr, y_tr).predict_proba(X_te)[:, 1]
print('TEST AUC', roc_auc_score(y_te, p), 'AP', average_precision_score(y_te, p), '(prevalence %.3f)' % y_te.mean())
print('TEST F1 at chosen threshold', f1_score(y_te, p >= thr)); print(confusion_matrix(y_te, p >= thr))
print('baseline AUC', roc_auc_score(y_te, base), 'AP', average_precision_score(y_te, base))